Lab 4: LLMs and Prompt Engineering for Decision Support
Duration: 2 weeks [30 Jul - 13 Aug, 2026] Due Date: 13th August, 2026 Format: Jupyter Notebook / Google Colab + external APIs + GitHub version control Grading: This is a graded lab.

Student Name: Louisa-Lois Student ID: 13532028

Part 0: Repository and API-key setup

In [2]:
!git clone https://github.com/Louisa-Lois/lab-4-llm-decision-support.git
%cd lab-4-llm-decision-support

Cloning into 'lab-4-llm-decision-support'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 20 (delta 3), reused 18 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 18.88 KiB | 9.44 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/lab-4-llm-decision-support


In [3]:
!pip install openai -q

In [4]:
# API-key setup

import os
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")

# OpenAI-compatible client
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "openai/gpt-oss-120b"

print("Client ready.")

Client ready.


Section 1 — Talking to an LLM Programmatically

Part 1.1 — Your first API call

In [5]:
# Part 1.1: Your first API call

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


# Call it once with a simple question and print the answer
answer = ask_llm("What is microfinance, in one sentence?")
print("Answer:")
print(answer)

Answer:
Microfinance is the provision of small‑scale financial services—such as loans, savings, and insurance—to low‑income individuals or groups who lack access to traditional banking.


In [6]:
# Print response.usage as well
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is microfinance, in one sentence?"},
    ],
    temperature=0.7,
    max_tokens=500,
)

print("\nToken usage:")
print(response.usage)


Token usage:
CompletionUsage(completion_tokens=67, prompt_tokens=89, total_tokens=156, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=29, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.017298704, prompt_time=0.003322875, completion_time=0.13992438, total_time=0.143247255)


**Student Reasoning - Anatomy of a call**

**1. Difference between system and user roles:**
The system role sets the model's overall behaviour, persona, and rules for the
entire conversation, it's set once and shapes how the model responds to
everything that follows. The user role is the actual question or task being
asked right now. Example: system = "You are a helpful assistant" (sets the
general behaviour), user = "What is microfinance, in one sentence?" (the
specific request). In this lab, the system role will later be used to give
the model a specific job (e.g. "You are an assistant to a microfinance loan
officer...") while the user role delivers the actual letter to process.

**2. What is a token, and why bill per token?**
A token is roughly a piece of a word, sometimes a whole word, sometimes
part of one. My test call used 89 prompt tokens (the question + system
message) and 76 completion tokens (the answer), totaling 165 tokens. API
providers bill per token rather than per request because the actual
computational cost of generating a response scales directly with how much
text is processed and produced, a one-word answer costs far less compute
than a 500-word essay, even though both are "one request." Billing per
token ties cost directly to actual resource usage.

Part 1.2 — Temperature: the randomness dial

In [7]:
# Part 1.2: Temperature — the randomness dial

question = "Suggest a name for a savings product for market traders in Accra."

# 5 calls at temperature = 0.0
print("=== Temperature = 0.0 ===\n")
answers_temp0 = []
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    answers_temp0.append(answer)
    print(f"Run {i+1}: {answer}\n")

# calls at temperature = 1.2
print("\n=== Temperature = 1.2 ===\n")
answers_temp12 = []
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    answers_temp12.append(answer)
    print(f"Run {i+1}: {answer}\n")

=== Temperature = 0.0 ===

Run 1: **Name Ideas for a Savings Product Targeted at Market Traders in Accra**

| # | Suggested Name | Why It Works |
|---|----------------|--------------|
| 1 | **“Kokro‑Kokro Savings”** | *Kokro* means “small” in Twi, emphasizing that even modest daily earnings can grow when saved consistently. |
| 2 | **“Bɔkɔɔ Bank”** | *Bɔkɔɔ* translates to “steady” or “smooth,” conveying a hassle‑free, reliable way to set aside money. |
| 3 | **“Market‑Mogya Fund”** | *Mogya* means “blood” – the lifeblood of the market. The name ties the product directly to traders’ daily hustle. |
| 4 | **“Adwuma Nest”** | *Adwuma* = “work” or “business.” A “nest” evokes safety and future growth for earnings from the market. |
| 5 | **“Sika Sika Savings”** | Repeating *sika* (“money”) creates a rhythmic, memorable brand that feels like a chant you’d hear in the market. |
| 6 | **“Kente Kash”** | Kente is a beloved Ghanaian fabric pattern; pairing it with “Kash” signals pride, heritage,

**Student Reasoning - Temperature**

At temperature 0.0, the answers were mostly repeated, Runs 1, 3, and 5
gave nearly identical lists (same names: Kokro-Kokro Savings, Bɔkɔɔ Bank,
Kente Kash...), and Runs 2 an                                                                                                                                                            d 4 were identical to each other but
different from 1/3/5. This shows temp=0.0 makes the model pick the most
likely response almost every time, though it's not perfectly deterministic.

At temperature 1.2, every single run gave a completely different set of
names and even different formats (some had taglines, one focused on a
single name instead of a list). This shows higher temperature makes the
model much more random and creative.

For the loan decision-support system, temperature=0.0 is the right
choice. The system needs to extract facts and give consistent advice
from a loan letter, not be creative. A loan officer needs the same
letter to produce the same summary and risk assessment every time, not
a different answer depending on luck.

Section 2 — The Dataset: Loan Application Letters

In [8]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


Section 3 — Prompt Engineering for the Decision Support System

Part 3.1 — Component 1: Summarization

In [9]:
# Part 3.1: Component 1 — Summarization

# V1: naive prompt
SUMMARY_PROMPT_V1 = "Summarize this:"

print("=== SUMMARY V1 — L002 ===\n")
v1_l002 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}", temperature=0)
print(v1_l002)

print("\n\n=== SUMMARY V1 — L006 ===\n")
v1_l006 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L006']}", temperature=0)
print(v1_l006)

=== SUMMARY V1 — L002 ===

Kwame Boateng, a commercial driver in Kumasi, is requesting an urgent loan of GHS 25,000 to repair his trotro engine and cover personal debts. He notes that business has been slow but expects improvement after the festive season, and he can repay the loan once funds become available, though he currently has no collateral. He is asking for quick assistance.


=== SUMMARY V1 — L006 ===

**Summary**

Kofi, a 22‑year‑old entrepreneur, is seeking a GHS 50,000 loan to launch three ventures—a car‑washing service, a provision shop, and an import business for phones from Dubai. He has no collateral but claims to be trustworthy and confident that the businesses will generate enough profit to repay the loan within one year. His friends describe him as business‑minded, though he has not yet started any of the enterprises.


In [10]:
# V2: engineered prompt with role + constraints

SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer in Ghana.
Your job is to summarize loan application letters into short, factual briefs
the officer can scan quickly.

Rules:
- Write exactly 3-4 sentences.
- Be strictly factual and neutral in tone — do not add opinions or judgments.
- Do NOT invent, assume, or infer any detail that is not explicitly stated in the letter.
- If a key detail (amount, purpose, repayment plan) is missing or vague, say so plainly
  rather than filling it in.
"""

def SUMMARY_PROMPT_V2(letter_text):
    return f"Summarize this loan application:\n\n{letter_text}"


print("=== SUMMARY V2 — L002 ===\n")
v2_l002 = ask_llm(
    SUMMARY_PROMPT_V2(LETTERS['L002']),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0
)
print(v2_l002)

print("\n\n=== SUMMARY V2 — L006 ===\n")
v2_l006 = ask_llm(
    SUMMARY_PROMPT_V2(LETTERS['L006']),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0
)
print(v2_l006)

=== SUMMARY V2 — L002 ===

Kwame Boateng, a commercial driver based in Kumasi, is applying for a loan of GHS 25,000. He states the funds are needed to repair his trotro engine and to settle personal debts. He indicates he has no collateral and that repayment will be made “whenever the money comes,” without specifying a repayment schedule or timeline. He requests a quick disbursement.


=== SUMMARY V2 — L006 ===

Kofi requests a loan of GHS 50,000. He intends to use the funds to start a car‑washing business, a provision shop, and to import phones from Dubai. He is 22 years old and says he will repay the loan in one year when the businesses are booming, but provides no detailed repayment schedule. No collateral is offered.


In [11]:
# Side-by-side comparison

print("=" * 70)
print("V1 vs V2 COMPARISON — L002")
print("=" * 70)
print(f"\nV1:\n{v1_l002}")
print(f"\nV2:\n{v2_l002}")

print("\n\n" + "=" * 70)
print("V1 vs V2 COMPARISON — L006")
print("=" * 70)
print(f"\nV1:\n{v1_l006}")
print(f"\nV2:\n{v2_l006}")

V1 vs V2 COMPARISON — L002

V1:
Kwame Boateng, a commercial driver in Kumasi, is requesting an urgent loan of GHS 25,000 to repair his trotro engine and cover personal debts. He notes that business has been slow but expects improvement after the festive season, and he can repay the loan once funds become available, though he currently has no collateral. He is asking for quick assistance.

V2:
Kwame Boateng, a commercial driver based in Kumasi, is applying for a loan of GHS 25,000. He states the funds are needed to repair his trotro engine and to settle personal debts. He indicates he has no collateral and that repayment will be made “whenever the money comes,” without specifying a repayment schedule or timeline. He requests a quick disbursement.


V1 vs V2 COMPARISON — L006

V1:
**Summary**

Kofi, a 22‑year‑old entrepreneur, is seeking a GHS 50,000 loan to launch three ventures—a car‑washing service, a provision shop, and an import business for phones from Dubai. He has no collateral b

**Student Reasoning — Summarization prompts**

**1. Problems in V1 that V2 fixed:**

V1 added a "**Summary**" header, inconsistent formatting for something meant
to be scanned quickly. V1 also used more interpretive language: it called
Kofi "a 22-year-old entrepreneur" and said he was "confident that the
businesses will generate enough profit to repay the loan", these are
judgments/characterizations not stated in the letter that way, just his own
claims being reported as fact. V2 fixed this by sticking to neutral phrasing
like "He states..." and "He intends to..."

V1 also didn't clearly flag missing information. For L002, V1 just said
"he promises to repay the loan as soon as funds become available" without
noting that's a red flag. V2 explicitly wrote "without specifying a
repayment schedule or timeline", directly surfacing the gap for the
loan officer instead of just restating the vague promise.

**2. Why "no invented details" matters, and its name in the literature:**

This failure mode is called hallucination, when an LLM states something
confidently that isn't actually supported by the source text. For a loan
officer, a hallucinated detail (like a wrong repayment amount or an
invented collateral claim) could directly lead to a bad lending decision.
Since the summary is meant to replace reading the full letter, any
invented detail becomes invisible to the officer, they have no way to
catch the error without re-reading the original letter themselves,
defeating the entire purpose of summarization.